# 05_Factor_Stability — 因子穩定性年度分析

## 目的

Phase 3-A 核心研究：找出在**所有年份都穩定有效**的因子。

**背景**：在 Phase 2 的 Fold-Internal IC 分析中，發現以下因子穩定性差異：

| 因子 | Fold 選中次數 (out of 24) | 狀態 |
|------|--------------------------|------|
| `vol_ratio` | 24/24 | 非常穩定 |
| `mom_1m_ra` | 23/24 | 非常穩定 |
| `trust_net_10d` | 22/24 | 穩定 |
| `mom_1m` | 21/24 | 穩定 |
| `price_52w` | **8/24** | **不穩定** |
| `foreign_net_20d` | **3/24** | **幾乎無效** |

本 Notebook 進一步按**年份**分解每個因子的 ICIR，找出：
- 方向每年一致的因子 → 可信賴的 alpha 信號
- 某年正/某年負的因子 → 噪音，不應依賴

## 自包含設計

此 Notebook 直接從 `finmind_cache/` 載入資料，**不需要先跑 01~04**。

In [ ]:
import os, sys, glob, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import spearmanr
from tqdm import tqdm

CACHE = '../finmind_cache'
print('Loading data from cache...')

In [ ]:
# ── 資料載入（與 strategy.py 相同流程）
close = pd.read_pickle(os.path.join(CACHE, 'close_wide.pkl'))
close.index = pd.to_datetime(close.index)
close.index.name = 'date'
close.columns.name = 'stock_id'

# High / Low / Money（用於 ATR 和流動性）
high_frames, low_frames, money_frames = [], [], []
for f in tqdm(glob.glob(os.path.join(CACHE, 'price/*.pkl')), desc='OHLCV'):
    try:
        df = pd.read_pickle(f)
        df['date'] = pd.to_datetime(df['date'])
        for col, lst in [('max', high_frames), ('min', low_frames), ('Trading_money', money_frames)]:
            if col in df.columns:
                tmp = df[['date', 'stock_id', col]].copy()
                tmp[col] = pd.to_numeric(tmp[col], errors='coerce')
                lst.append(tmp)
    except Exception:
        pass

def _wide(frames, col):
    df = pd.concat(frames, ignore_index=True)
    w  = df.pivot_table(index='date', columns='stock_id', values=col)
    w.index = pd.to_datetime(w.index); w.index.name = 'date'
    return w

high_wide  = _wide(high_frames,  'max')
low_wide   = _wide(low_frames,   'min')
money_wide = _wide(money_frames, 'Trading_money')

# 月營收 + 三大法人
rev_frames, fg_frames, tr_frames = [], [], []
for f in tqdm(glob.glob(os.path.join(CACHE, 'revenue/*.pkl')), desc='Revenue'):
    try:
        df = pd.read_pickle(f)
        df['date'] = pd.to_datetime(df['date'])
        rev_frames.append(df[['date', 'stock_id', 'revenue']])
    except Exception: pass

for f in tqdm(glob.glob(os.path.join(CACHE, 'institution/*.pkl')), desc='Institution'):
    try:
        df = pd.read_pickle(f)
        df['date'] = pd.to_datetime(df['date'])
        df['net'] = df['buy'] - df['sell']
        fg_frames.append(df[df['name'] == 'Foreign_Investor'][['date', 'stock_id', 'net']])
        tr_frames.append(df[df['name'] == 'Investment_Trust'][['date', 'stock_id', 'net']])
    except Exception: pass

rev_wide    = _wide(rev_frames,  'revenue')
foreign_net = _wide(fg_frames,   'net')
trust_net   = _wide(tr_frames,   'net')

print(f'close: {close.shape} | high: {high_wide.shape} | rev: {rev_wide.shape}')

In [ ]:
# ── 因子計算（與 strategy.py 相同）
def resample_monthly(df):
    return df.resample('ME').last()

def zscore_xs(df):
    df = df.replace([np.inf, -np.inf], np.nan)
    mu  = df.mean(axis=1)
    std = df.std(axis=1).replace(0, np.nan)
    return (df.sub(mu, axis=0).div(std, axis=0)).clip(-3, 3)

def to_long(df, name):
    s = df.unstack().swaplevel().sort_index()
    s.name = name
    return s

mom_1m    = close / close.shift(21) - 1
mom_3m    = close / close.shift(63) - 1
mom_6m    = close / close.shift(126) - 1
rvol_20   = close.pct_change().rolling(20).std().replace(0, np.nan)
mom_1m_ra = (mom_1m / rvol_20).replace([np.inf, -np.inf], np.nan)
high_52w  = close.rolling(252).max()
price_52w = close / high_52w.replace(0, np.nan)
high_20d  = close.rolling(20).max()
breakout  = (close / high_20d.replace(0, np.nan) - 1)

def calc_rsi(price_df, period=14):
    delta = price_df.diff()
    gain  = delta.clip(lower=0).ewm(com=period-1, min_periods=period).mean()
    loss  = (-delta).clip(lower=0).ewm(com=period-1, min_periods=period).mean()
    return 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

rsi_14    = calc_rsi(close)
hl        = (high_wide - low_wide).abs()
atr_14    = hl.rolling(14).mean()
atr_rel   = (atr_14 / close.replace(0, np.nan))
vol_ratio = money_wide.rolling(20).mean() / money_wide.rolling(60).mean()

rev_yoy   = rev_wide.pct_change(12)
rev_mom   = rev_wide.pct_change(1)
rev_accel = rev_wide.pct_change(1) - rev_wide.pct_change(1).shift(3)

rev_yoy_d   = rev_yoy.reindex(close.index, method='ffill')
rev_mom_d   = rev_mom.reindex(close.index, method='ffill')
rev_accel_d = rev_accel.reindex(close.index, method='ffill')

vol_sum20       = money_wide.rolling(20).sum()
common_f        = foreign_net.columns.intersection(vol_sum20.columns)
common_t        = trust_net.columns.intersection(vol_sum20.columns)
foreign_net_20d = foreign_net[common_f].rolling(20).sum() / vol_sum20[common_f].replace(0, np.nan)
trust_net_10d   = trust_net[common_t].rolling(10).sum()  / vol_sum20[common_t].replace(0, np.nan)

factor_defs = {
    'mom_1m'         : resample_monthly(mom_1m),
    'mom_3m'         : resample_monthly(mom_3m),
    'mom_6m'         : resample_monthly(mom_6m),
    'rsi_14'         : resample_monthly(rsi_14),
    'vol_ratio'      : resample_monthly(vol_ratio),
    'rev_yoy'        : resample_monthly(rev_yoy_d),
    'rev_mom'        : resample_monthly(rev_mom_d),
    'foreign_net_20d': resample_monthly(foreign_net_20d),
    'trust_net_10d'  : resample_monthly(trust_net_10d),
    'mom_1m_ra'      : resample_monthly(mom_1m_ra),
    'price_52w'      : resample_monthly(price_52w),
    'breakout'       : resample_monthly(breakout),
    'atr_rel'        : resample_monthly(atr_rel),
    'rev_accel'      : resample_monthly(rev_accel_d),
}

factors_z    = {name: zscore_xs(df) for name, df in factor_defs.items()}
combined_long = [to_long(df, name) for name, df in factors_z.items()]
X_raw = pd.concat(combined_long, axis=1)

# 目標值
close_m     = resample_monthly(close)
monthly_ret = close_m.pct_change(1).shift(-1)
mkt_med     = monthly_ret.median(axis=1)
excess_ret  = monthly_ret.subtract(mkt_med, axis=0)
y_raw       = to_long(excess_ret, 'excess_return')

X_raw, y_raw = X_raw.align(y_raw, join='inner', axis=0)
X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
y_raw = y_raw.replace([np.inf, -np.inf], np.nan)
mask  = X_raw.notna().all(axis=1) & y_raw.notna()
X, y  = X_raw[mask], y_raw[mask]

all_dates  = X.index.get_level_values(0).unique().sort_values()
TEST_START = pd.Timestamp('2020-01-01')
print(f'X shape: {X.shape} | Date: {all_dates[0].date()} ~ {all_dates[-1].date()}')

In [ ]:
# ── 按年計算每個因子的月度 IC，整理成熱力圖格式
# 只使用 TEST_START 之後的資料（避免訓練期資料汙染分析）

test_dates = [d for d in all_dates if d >= TEST_START]
years      = sorted(set(d.year for d in test_dates))

ic_by_year = {}  # {factor: {year: [monthly_IC values]}}

for factor in tqdm(X.columns, desc='Computing IC by year'):
    ic_by_year[factor] = {yr: [] for yr in years}
    for d in test_dates:
        try:
            xf = X.xs(d, level=0)[factor].dropna()
            yf = y.xs(d, level=0).reindex(xf.index).dropna()
            xf = xf.reindex(yf.index)
            if len(yf) > 10:
                ic_val, _ = spearmanr(xf.values, yf.values)
                if not np.isnan(ic_val):
                    ic_by_year[factor][d.year].append(ic_val)
        except Exception:
            pass

print('IC computation done')

In [ ]:
# ── 彙整成 ICIR 熱力圖資料
icir_matrix = {}  # {factor: {year: ICIR}}
ic_mean_mat = {}  # for display

for factor in X.columns:
    icir_matrix[factor] = {}
    ic_mean_mat[factor] = {}
    for yr in years:
        ics = pd.Series(ic_by_year[factor][yr])
        if len(ics) >= 3:
            icir  = ics.mean() / (ics.std() + 1e-8)
            ic_mn = ics.mean()
        else:
            icir, ic_mn = np.nan, np.nan
        icir_matrix[factor][yr] = icir
        ic_mean_mat[factor][yr] = ic_mn

icir_df  = pd.DataFrame(icir_matrix).T    # rows=factor, cols=year
icmean_df = pd.DataFrame(ic_mean_mat).T

# 排序：依全期平均 |ICIR| 降序
overall_abs_icir = icir_df.abs().mean(axis=1).sort_values(ascending=False)
icir_df   = icir_df.loc[overall_abs_icir.index]
icmean_df = icmean_df.loc[overall_abs_icir.index]

print('ICIR Heatmap data (row=factor, col=year):')
display(icir_df.round(3))

In [ ]:
# ── 視覺化：ICIR 年度熱力圖
fig = go.Figure(go.Heatmap(
    z           = icir_df.values,
    x           = [str(y) for y in icir_df.columns],
    y           = icir_df.index.tolist(),
    colorscale  = 'RdBu',
    zmid        = 0,
    text        = icir_df.round(2).values,
    texttemplate= '%{text}',
    showscale   = True,
    colorbar    = dict(title='ICIR'),
))
fig.update_layout(
    title      = 'Factor ICIR by Year (2020~2026) — Red=Bearish factor, Blue=Bullish factor',
    xaxis_title= 'Year',
    yaxis_title= 'Factor',
    template   = 'plotly_dark',
    height     = 550,
)
fig.show()
print('閱讀指引：藍色=該年因子有效（負值因子用於空頭），紅色=反向，白色=無效')
print('理想的因子：每年都是同色（方向一致），不出現顏色在年份間切換')

In [ ]:
# ── 因子穩定性評分：方向一致性（sign consistency）
# 指標定義：每個因子的 ICIR 在所有年份中，符號相同的比例
# 若全年都是 ICIR<0 → 100%（穩定反向因子）
# 若全年都是 ICIR>0 → 100%（穩定正向因子）
# 若某年正某年負 → 低%（不穩定，不可依賴）

stability_report = []
for factor in icir_df.index:
    vals = icir_df.loc[factor].dropna()
    if len(vals) == 0: continue
    pct_negative = (vals < 0).mean()
    pct_positive = (vals > 0).mean()
    sign_consistency = max(pct_negative, pct_positive)
    avg_abs_icir     = vals.abs().mean()
    direction        = 'NEGATIVE (contrarian)' if pct_negative > pct_positive else 'POSITIVE (momentum)'
    stability_report.append({
        'Factor'           : factor,
        'Direction'        : direction,
        'Sign Consistency' : f'{sign_consistency:.0%}',
        'Avg |ICIR|'       : f'{avg_abs_icir:.3f}',
        'Recommendation'   : 'KEEP' if sign_consistency >= 0.80 else ('REVIEW' if sign_consistency >= 0.60 else 'DROP'),
    })

stability_df = pd.DataFrame(stability_report).set_index('Factor')
print('Factor Stability Report:')
display(stability_df)

In [ ]:
# ── ICIR 逐年折線圖（每個因子一條線）
keep_factors   = stability_df[stability_df['Recommendation'] == 'KEEP'].index.tolist()
review_factors = stability_df[stability_df['Recommendation'] == 'REVIEW'].index.tolist()
drop_factors   = stability_df[stability_df['Recommendation'] == 'DROP'].index.tolist()

fig2 = go.Figure()
year_labels = [str(y) for y in icir_df.columns]

for factor in icir_df.index:
    rec = stability_df.loc[factor, 'Recommendation']
    color = {'KEEP': '#2962FF', 'REVIEW': '#FFA000', 'DROP': '#E53935'}[rec]
    dash  = {'KEEP': 'solid', 'REVIEW': 'dash', 'DROP': 'dot'}[rec]
    fig2.add_trace(go.Scatter(
        x=year_labels, y=icir_df.loc[factor].values,
        name=f'{factor} ({rec})', mode='lines+markers',
        line=dict(color=color, dash=dash, width=2),
    ))

fig2.add_hline(y=0, line_color='white', line_dash='dash', line_width=1)
fig2.update_layout(
    title   = 'Factor ICIR Trend by Year — Blue=KEEP, Orange=REVIEW, Red=DROP',
    xaxis_title='Year', yaxis_title='ICIR',
    template='plotly_dark', height=500,
    legend=dict(x=1.01, y=0.99),
)
fig2.show()

In [ ]:
# ── 最終建議
print('='*60)
print('PHASE 3-A 結論：因子篩選建議')
print('='*60)
print(f'\n  KEEP  ({len(keep_factors)} 個）：{keep_factors}')
print(f'  REVIEW ({len(review_factors)} 個）：{review_factors}')
print(f'  DROP  ({len(drop_factors)} 個）：{drop_factors}')

print('\n下一步建議：')
print('  1. 在 strategy.py 中移除 DROP 的因子，只保留 KEEP + REVIEW 進入 Fold-Internal IC')
print('  2. 執行 strategy.py 觀察移除不穩定因子後的 CAGR 變化')
print('  3. 接著執行 06_Portfolio_Size_Sweep.ipynb（持股數量掃描）')